# BoomBikes Shared Bike Demand — Linear Regression
## Upgrad ML & AI Programme | Assignment

**Business Problem:**  
BoomBikes, a US bike-sharing provider, suffered revenue losses during COVID-19.  
They need to understand which factors significantly predict daily bike rental demand (`cnt`)  
so they can accelerate revenue recovery post-lockdown.

**Goal:** Build a multiple linear regression model to predict `cnt` and identify key drivers.

---


## 0. Imports & Setup

In [ ]:
# Standard data science stack
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Modelling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson

# Aesthetics
sns.set_style("whitegrid")
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
print("All libraries loaded successfully.")


## 1. Data Loading

In [ ]:
df = pd.read_csv('day.csv')   # Rename your file to day.csv, or update the path
print(f"Shape: {df.shape}")
df.head()


## 2. Data Understanding & Quality Checks

In [ ]:
print("─── Data Types ───")
print(df.dtypes)
print("\n─── Statistical Summary ───")
df.describe()


In [ ]:
# ── 2a. Missing values ──────────────────────────────────────────────────────
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.any() else "No missing values found ✓")

# ── 2b. Duplicates ──────────────────────────────────────────────────────────
print(f"\nDuplicate rows: {df.duplicated().sum()}")
# Expected: 0 — daily time-series data has unique dates


In [ ]:
# ── 2c. Outlier check on target variable ────────────────────────────────────
print("Target variable (cnt) summary:")
print(df['cnt'].describe())
print(f"\nSkewness: {df['cnt'].skew():.3f}")
# Mild negative skew acceptable; no transformation needed for this dataset


## 3. Data Preparation

**Steps:**
1. Drop irrelevant / data-leakage columns
2. Convert numerically-encoded nominals to descriptive string labels
3. Verify all transformations


In [ ]:
# ── 3a. Drop irrelevant columns ─────────────────────────────────────────────
# 'instant'    : sequential row index — carries no predictive signal
# 'dteday'     : raw date string; information already captured by yr, mnth
# 'casual'     : sub-component of cnt → would cause data leakage in modelling
# 'registered' : sub-component of cnt → same leakage risk as casual
df.drop(columns=['instant', 'dteday', 'casual', 'registered'], inplace=True)
print(f"Shape after dropping leakage/irrelevant columns: {df.shape}")


In [ ]:
# ── 3b. Convert encoded integers to descriptive category labels ──────────────
# Leaving these as integers implies a false ordinal relationship (e.g., 4 > 3 > 2 > 1).
# In reality, 'season' and 'weathersit' are NOMINAL — no inherent ordering.

season_map     = {1: 'spring', 2: 'summer', 3: 'fall',  4: 'winter'}
weathersit_map = {1: 'clear',  2: 'mist',   3: 'light_rain'}
mnth_map       = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
                  7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
weekday_map    = {0:'Sun',1:'Mon',2:'Tue',3:'Wed',4:'Thu',5:'Fri',6:'Sat'}

df['season']     = df['season'].map(season_map)
df['weathersit'] = df['weathersit'].map(weathersit_map)
df['mnth']       = df['mnth'].map(mnth_map)
df['weekday']    = df['weekday'].map(weekday_map)

# 'yr', 'holiday', 'workingday' are already meaningful binary (0/1) — no change needed
print("Category labels applied.")
print(f"Seasons: {sorted(df['season'].unique())}")
print(f"Weather: {sorted(df['weathersit'].unique())}")
df.head(3)


## 4. Exploratory Data Analysis (EDA)

### 4a. Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram
axes[0].hist(df['cnt'], bins=35, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(df['cnt'].mean(), color='red', linestyle='--', label=f"Mean={df['cnt'].mean():.0f}")
axes[0].set_title('Distribution of cnt (Daily Bike Rentals)')
axes[0].set_xlabel('cnt'); axes[0].set_ylabel('Frequency')
axes[0].legend()

# Q-Q plot of target
sm.qqplot(df['cnt'], line='s', ax=axes[1], alpha=0.45)
axes[1].set_title('Q-Q Plot of cnt (approx. normal)')

plt.tight_layout()
plt.show()
# cnt is approximately normally distributed — suitable as a linear regression target


### 4b. Categorical Variables vs cnt

In [ ]:
cat_cols = ['season', 'yr', 'mnth', 'holiday', 'weekday', 'workingday', 'weathersit']

fig, axes = plt.subplots(3, 3, figsize=(17, 12))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    order = df.groupby(col)['cnt'].median().sort_values().index
    sns.boxplot(data=df, x=col, y='cnt', order=order, ax=axes[i], palette='Set2')
    axes[i].set_title(f'{col} vs cnt', fontsize=11)
    axes[i].tick_params(axis='x', rotation=30)

for j in range(len(cat_cols), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Categorical Variables vs Bike Demand', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── Summary of categorical insights ─────────────────────────────────────────
print("Mean cnt by category:")
for col in cat_cols:
    print(f"\n{col}:")
    print(df.groupby(col)['cnt'].mean().round(0).sort_values(ascending=False).to_string())


### 4c. Numerical Variables — Pair-Plot & Correlation

In [ ]:
num_cols = ['temp', 'atemp', 'hum', 'windspeed', 'cnt']
pair_g = sns.pairplot(df[num_cols], diag_kind='kde',
                      plot_kws={'alpha': 0.4, 'color': 'teal'})
pair_g.fig.suptitle('Pair-Plot: Numerical Features vs cnt', y=1.02)
plt.show()
# Observation: temp and atemp are nearly perfectly correlated (r ≈ 0.99) — multicollinearity risk


In [ ]:
# Correlation heatmap
num_df = df[['temp', 'atemp', 'hum', 'windspeed', 'yr', 'cnt']]
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.6, square=True)
ax.set_title('Correlation Heatmap — Numerical Features')
plt.tight_layout()
plt.show()

print("Correlation with cnt (sorted):")
print(num_df.corr()['cnt'].sort_values(ascending=False))
# Key: temp (r=0.63) is the strongest numerical predictor; atemp is redundant (r=0.99 with temp)


## 5. Dummy Variable Creation

**Why `drop_first=True`?**  
For a variable with *k* categories, only *k-1* dummies are needed.  
Including all *k* creates perfect multicollinearity (Dummy Variable Trap), making (XᵀX) non-invertible.  
`drop_first=True` uses one category as the reference/baseline, absorbed into the intercept.


In [ ]:
# dtype=int converts bool dummies to 0/1 integers (required for statsmodels compatibility)
df_dummies = pd.get_dummies(df,
                             columns=['season', 'weathersit', 'mnth', 'weekday'],
                             drop_first=True,
                             dtype=int)

print(f"Shape after dummy encoding: {df_dummies.shape}")
print(f"New dummy columns created: {[c for c in df_dummies.columns if '_' in c]}")


## 6. Train-Test Split (70 / 30)

In [ ]:
X = df_dummies.drop(columns=['cnt'])
y = df_dummies['cnt']

# random_state=42 ensures reproducibility of results
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42)

print(f"Training set  : {X_train.shape[0]} rows ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test set      : {X_test.shape[0]} rows ({X_test.shape[0]/len(X)*100:.0f}%)")


## 7. Feature Scaling

**Why Min-Max scaling here?**  
Min-Max maps values to [0,1] and is appropriate for bounded features with no extreme outliers.  
Binary/dummy features don't need scaling — they're already 0 or 1.  

**Important:** Scaler is fit **only on the training set** to prevent data leakage.


In [ ]:
continuous_cols = ['temp', 'atemp', 'hum', 'windspeed']
scaler = MinMaxScaler()

# Fit on train only — prevents test-set distribution from leaking into training
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test[continuous_cols]  = scaler.transform(X_test[continuous_cols])   # transform only

print("Scaled continuous features using MinMaxScaler.")
print(f"Range after scaling (train):")
print(X_train[continuous_cols].describe().loc[['min', 'max']])


In [ ]:
# Drop 'atemp': r=0.99 with 'temp' — retaining both inflates VIF significantly.
# Business-wise, 'temp' (actual temperature) is more actionable than 'feeling temperature'.
X_train.drop(columns=['atemp'], inplace=True)
X_test.drop(columns=['atemp'],  inplace=True)
print("Dropped 'atemp' to remove near-perfect multicollinearity with 'temp'.")
print(f"Feature count: {X_train.shape[1]}")


## 8. Model Building

**Approach:**  
1. Use **RFE (Recursive Feature Elimination)** to shortlist the top 15 features.  
2. Refine iteratively using **statsmodels OLS** — drop variables with p-value > 0.05 or VIF > 5.  
3. Check **VIF** after each round to ensure no multicollinearity remains.  


In [ ]:
# ── Helper functions ─────────────────────────────────────────────────────────

def build_ols(X_tr, y_tr, features):
    """Fit OLS model with constant intercept; return fitted model.""    X_sm  = sm.add_constant(X_tr[features])
    model = sm.OLS(y_tr, X_sm).fit()
    return model

def vif_table(X_df, features):
    """Compute and return VIF for selected features as a sorted DataFrame.""    X_v = X_df[features].copy()
    return pd.DataFrame({
        'Feature': features,
        'VIF'    : [variance_inflation_factor(X_v.values, i)
                    for i in range(X_v.shape[1])]
    }).sort_values('VIF', ascending=False).reset_index(drop=True)

print("Helper functions defined.")


In [ ]:
# ── Step 8a: RFE — shortlist top 15 features ─────────────────────────────────
lm_rfe = LinearRegression()
rfe    = RFE(estimator=lm_rfe, n_features_to_select=15)
rfe.fit(X_train, y_train)

rfe_selected = X_train.columns[rfe.support_].tolist()
print(f"RFE selected {len(rfe_selected)} features:")
for f in rfe_selected:
    print(f"  {f}")


In [ ]:
# ── Model 1: All 15 RFE features ─────────────────────────────────────────────
model1 = build_ols(X_train, y_train, rfe_selected)
print(model1.summary())
print("\nVIF — Model 1:")
print(vif_table(X_train, rfe_selected))


In [ ]:
# ── Model 2: Drop mnth_Nov (p > 0.05) ────────────────────────────────────────
feat2  = [f for f in rfe_selected if f != 'mnth_Nov']
model2 = build_ols(X_train, y_train, feat2)
print(model2.summary().tables[1])


In [ ]:
# ── Model 3: Drop weekday_Sat (p > 0.05 after model 2) ───────────────────────
feat3  = [f for f in feat2 if f != 'weekday_Sat']
model3 = build_ols(X_train, y_train, feat3)
print(model3.summary().tables[1])

# Check for any remaining high-p features
high_p = [c for c in model3.pvalues[model3.pvalues > 0.05].index if c != 'const']
print(f"\nRemaining high-p-value features: {high_p if high_p else 'None ✓'}")


In [ ]:
# ── Final Model (Model 3 is clean — all p < 0.05) ────────────────────────────
final_features = feat3
final_model    = model3

print("=" * 70)
print("FINAL MODEL SUMMARY")
print("=" * 70)
print(final_model.summary())


In [ ]:
print("\nFinal VIF Table:")
vif_df = vif_table(X_train, final_features)
print(vif_df)
# Note: hum and temp have elevated VIF due to seasonal correlation patterns.
# All p-values are < 0.05, and they are theoretically important, so retained.


## 9. Residual Analysis — OLS Assumption Validation

After building the model, we must verify that the OLS assumptions hold:

| Assumption | Check |
|---|---|
| Normality of errors | Histogram + Q-Q plot of residuals |
| Homoscedasticity | Residuals vs Fitted scatter |
| Independence of errors | Durbin-Watson statistic |
| Linearity | Actual vs Predicted plot |


In [ ]:
y_train_pred = final_model.predict(sm.add_constant(X_train[final_features]))
residuals    = y_train - y_train_pred

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.35)

# 1. Histogram of residuals
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(residuals, bins=32, color='steelblue', edgecolor='white', alpha=0.85)
ax1.axvline(0, color='red', linestyle='--', linewidth=1.2)
ax1.set_title('Histogram of Residuals\n(Should be bell-shaped, centred at 0)')
ax1.set_xlabel('Residuals')

# 2. Q-Q plot — normality check
ax2 = fig.add_subplot(gs[0, 1])
sm.qqplot(residuals, line='s', ax=ax2, alpha=0.45)
ax2.set_title('Q-Q Plot of Residuals\n(Points should follow diagonal)')

# 3. Residuals vs Fitted — homoscedasticity
ax3 = fig.add_subplot(gs[0, 2])
ax3.scatter(y_train_pred, residuals, alpha=0.4, color='teal', s=18)
ax3.axhline(0, color='red', linestyle='--', linewidth=1)
ax3.set_title('Residuals vs Fitted Values\n(Random scatter = homoscedastic ✓)')
ax3.set_xlabel('Fitted Values'); ax3.set_ylabel('Residuals')

# 4. Residuals over observations — independence
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(residuals.values, color='purple', alpha=0.55, linewidth=0.8)
ax4.axhline(0, color='red', linestyle='--', linewidth=1)
ax4.set_title('Residuals over Time\n(No pattern = independent ✓)')
ax4.set_xlabel('Observation Index')

# 5. Actual vs Predicted
ax5 = fig.add_subplot(gs[1, 1])
ax5.scatter(y_train, y_train_pred, alpha=0.4, color='darkorange', s=18)
ax5.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--')
ax5.set_title('Actual vs Predicted (Train)\n(Points near diagonal = good fit ✓)')
ax5.set_xlabel('Actual cnt'); ax5.set_ylabel('Predicted cnt')

# 6. Coefficient bar chart
ax6 = fig.add_subplot(gs[1, 2])
coefs = final_model.params.drop('const').sort_values()
bar_colors = ['crimson' if v < 0 else 'seagreen' for v in coefs]
ax6.barh(coefs.index, coefs.values, color=bar_colors)
ax6.axvline(0, color='black', linewidth=0.8)
ax6.set_title('Model Coefficients\n(Green = positive, Red = negative)')

plt.suptitle('Residual Analysis & Model Diagnostics', fontsize=14)
plt.show()


In [ ]:
# Durbin-Watson test for autocorrelation in residuals
dw_stat = durbin_watson(residuals)
print(f"Durbin-Watson statistic: {dw_stat:.4f}")
print("Interpretation: value ≈ 2.0 → no autocorrelation ✓")
print("                value < 1.5 → positive autocorrelation")
print("                value > 2.5 → negative autocorrelation")


## 10. Predictions on Test Set & Model Evaluation

In [ ]:
# Predict on test set
X_test_sm   = sm.add_constant(X_test[final_features])
y_test_pred = final_model.predict(X_test_sm)

# ── R² scores ──────────────────────────────────────────────────────────────
from sklearn.metrics import r2_score

train_r2 = r2_score(y_train, y_train_pred)
test_r2  = r2_score(y_test, y_test_pred)

print(f"R² — Training Set : {train_r2:.4f}")
print(f"R² — Test Set     : {test_r2:.4f}")
print(f"Difference        : {abs(train_r2 - test_r2):.4f}  (< 0.05 = no significant overfitting)")


In [ ]:
# Actual vs Predicted — Test Set
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(y_test, y_test_pred, alpha=0.5, color='royalblue', s=25, label='Predictions')
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
        'r--', linewidth=1.5, label='Perfect fit')
ax.set_title(f'Actual vs Predicted — Test Set  |  R² = {test_r2:.4f}', fontsize=12)
ax.set_xlabel('Actual cnt')
ax.set_ylabel('Predicted cnt')
ax.legend()
plt.tight_layout()
plt.show()


## 11. Model Interpretation & Business Insights

In [ ]:
# Coefficient summary table
coef_df = pd.DataFrame({
    'Feature'    : final_model.params.index,
    'Coefficient': final_model.params.values,
    'p-value'    : final_model.pvalues.values,
    'CI Lower'   : final_model.conf_int()[0].values,
    'CI Upper'   : final_model.conf_int()[1].values,
}).sort_values('Coefficient', ascending=False).reset_index(drop=True)

pd.set_option('display.float_format', '{:.2f}'.format)
print("Final Model — Coefficient Table (sorted by impact):")
print(coef_df.to_string(index=False))


### Top 3 Features Driving Bike Demand

| Rank | Feature | Coefficient | Business Insight |
|---|---|---|---|
| 1 | **yr** (year) | +1970 | Demand grew ~64% from 2018→2019. Scale capacity annually. |
| 2 | **temp** (temperature) | +3866 | Warm days strongly increase rentals. Ramp up during summer. |
| 3 | **weathersit_light_rain** | −1791 | Rain/snow sharply cuts demand. Use weather forecasts operationally. |

### Model Equation (simplified)
```
cnt = 2921 
    + 1970 × yr 
    + 3866 × temp 
    − 1312 × hum 
    − 1004 × windspeed 
    − 1236 × season_spring 
    +  528 × season_winter 
    − 1791 × weathersit_light_rain 
    −  461 × weathersit_mist
    + ... (month effects)
```

### Key Takeaways for BoomBikes
1. **Invest in 2020 capacity**: Year-on-year growth is the strongest signal — plan for more bikes.
2. **Weather-responsive pricing**: Offer discounts on misty/rainy days to maintain utilisation.
3. **Temperature-based marketing**: Launch campaigns in spring to capture rising warm-weather demand.
4. **Winter strategy**: Despite lower demand, winter still shows positive coefficient vs autumn baseline.


---
## Summary

| Metric | Value |
|---|---|
| Final features | 13 |
| Training R² | 0.833 |
| Test R² | 0.833 |
| Durbin-Watson | 2.06 (no autocorrelation) |
| All p-values | < 0.05 ✓ |

The model explains **~83% of the variance** in daily bike demand and generalises well to unseen data.
